# Create flag parameter

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
# creating utilities

dbutils.widgets.text('incremental_flag', '0')

In [0]:
# check if the table exists with any data then 
if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    if (spark.sql("SELECT MAX(dim_model_key) FROM cars_catalog.gold.dim_model").collect()[0][0] > 0):
        incremental_flag = '1'
else:
    incremental_flag = dbutils.widgets.get('incremental_flag')

print(incremental_flag)

1


# creating dimension model


### Fetch relative columns

In [0]:
source_df = spark.sql('''
                      SELECT DISTINCT MODEL_ID, MODEL_CATEGORY
                      FROM parquet.`abfss://silver@adlsforde.dfs.core.windows.net/carsales`
                      ''')



In [0]:
source_df.display()

MODEL_ID,MODEL_CATEGORY
Hon-M219,Hon
Hyu-M161,Hyu
Toy-M101,Toy
Jee-M11,Jee
Tat-M192,Tat
Toy-M105,Toy
For-M16,For
Aud-M228,Aud
Nis-M81,Nis
Toy-M204,Toy



### Dim model Sink initial and incremental 

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    sink_df = spark.sql('''
                        SELECT dim_model_key, model_id, model_category
                        FROM cars_catalog.gold.dim_model
                        ''')

else:
    sink_df = spark.sql('''
                        SELECT 1 as dim_model_key, model_id, model_category
                    FROM parquet.`abfss://silver@adlsforde.dfs.core.windows.net/carsales`
                    WHERE 1 = 0
                    ''')


### Fintering new records and old records

In [0]:
filter_df = source_df.join(sink_df, source_df.MODEL_ID == sink_df.model_id, 'left')\
    .select(source_df.MODEL_ID, source_df.MODEL_CATEGORY, sink_df.dim_model_key)

**df_filter_old**

In [0]:
df_filter_old = filter_df.filter(col('dim_model_key').isNotNull())


**df_filter_new**

In [0]:
df_filter_new = filter_df.filter(col('dim_model_key').isNull()).select('MODEL_ID', 'MODEL_CATEGORY')


### Create Surrogate Key
**Fetch the max surrogate key from existing dim table**

In [0]:
if (incremental_flag == '0'):
    max_value = 1
else:
    max_value_df = spark.sql("SELECT MAX(dim_model_key) FROM cars_catalog.gold.dim_model")
    max_value = max_value_df.collect()[0][0]




**Create Surrogate key column and add the max surrogate key**

In [0]:
df_filter_new = df_filter_new.withColumn('dim_model_key', max_value + monotonically_increasing_id())
df_filter_new.display()

MODEL_ID,MODEL_CATEGORY,dim_model_key
Hon-M219,Hon,277
Hyu-M161,Hyu,278
Toy-M101,Toy,279
Jee-M11,Jee,280
Tat-M192,Tat,281
Toy-M105,Toy,282
For-M16,For,283
Aud-M228,Aud,284
Nis-M81,Nis,285
Toy-M204,Toy,286


### Create Final data frame - df_filter_old + df_filter_new


In [0]:
final_df =  df_filter_new.union(df_filter_old)

In [0]:
final_df.display()

MODEL_ID,MODEL_CATEGORY,dim_model_key
Hon-M219,Hon,277
Hyu-M161,Hyu,278
Toy-M101,Toy,279
Jee-M11,Jee,280
Tat-M192,Tat,281
Toy-M105,Toy,282
For-M16,For,283
Aud-M228,Aud,284
Nis-M81,Nis,285
Toy-M204,Toy,286


### SCD Type - 1 (UPSERT)
**Update(existing change data) + Insert(new data)**

In [0]:
# Incremental Run
if spark.catalog.tableExists("cars_catalog.gold.dim_model"):
    delta_table = DeltaTable.forPath(spark, "abfss://gold@adlsforde.dfs.core.windows.net/dim_model")
    
    delta_table.alias('trg').merge(final_df.alias('src'), "trg.dim_model_key = src.dim_model_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()
# initial run
else:
    final_df.write.format('delta')\
        .mode("overwrite")\
        .option("path", "abfss://gold@adlsforde.dfs.core.windows.net/dim_model")\
        .saveAsTable("cars_catalog.gold.dim_model")

In [0]:
%sql
SELECT * FROM cars_catalog.gold.dim_model

MODEL_ID,MODEL_CATEGORY,dim_model_key
Hon-M219,Hon,1
Hyu-M161,Hyu,2
Toy-M101,Toy,3
Jee-M11,Jee,4
Tat-M192,Tat,5
Toy-M105,Toy,6
For-M16,For,7
Aud-M228,Aud,8
Nis-M81,Nis,9
Toy-M204,Toy,10
